# 2.3 — ファイル・CSVの読み書き

これまでの値はプログラム終了時に失われました。このレッスンでは、外部ファイルに保存された図書記録を読み、検査し、別のCSVへ保存して、もう一度読み直せるところまで進みます。元データは上書きしません。

## 導入

このNotebookでは、本文の考え方を実際のコードで確かめます。

## このレッスンの到達目標

- 現在位置を仮定せず、定義した基準位置からパスを作れる。
- モードとUTF-8を明示し、安全にテキストファイルを開閉できる。
- DictReaderでCSVを読み、宣言したスキーマに従って値を変換できる。
- 使用前にヘッダーと各行を検証できる。
- DictWriterで別のCSVへ保存し、再読込した成果物を確認できる。

> **学習経路:** 必須：2.3.1～2.3.5　｜　統合練習：2.3.6


## 2.3.1 読み込むファイルの場所を確定する

相対パスは現在の作業フォルダを基準に解釈されます。Notebookを英語版の直下から開く場合と`ja`フォルダから開く場合では現在位置が異なるため、教材ファイルがある親フォルダまで順に探します。エラーになったら、推測でパスを書き換える前に、解決した絶対パスを表示します。

In [ ]:
from pathlib import Path

def find_course_file(relative_path):
    """Find a supplied course file from a Notebook opened at any course level."""
    start = Path.cwd().resolve()
    for folder in [start, *start.parents]:
        candidate = folder / relative_path
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"教材ファイルが見つかりません: {relative_path}; started at {start}")

source_path = find_course_file(Path("data") / "library-books-practice.csv")
print("Resolved source:", source_path)


### `.py`ファイルでは`__file__`を基準にする

Notebookでは`Path.cwd()`から教材ルートを探しました。独立したスクリプトでは、次の形でスクリプト自身の場所を基準にします。2.4のスタータープログラムでは、この形が最初から用意されています。

```python
BASE_DIR = Path(__file__).resolve().parent
INPUT_PATH = BASE_DIR / "data" / "books.csv"
OUTPUT_PATH = BASE_DIR / "output" / "books_updated.csv"
```

## 2.3.2 モードと文字コードを明示してファイルを開く

`with`ブロックを抜けるとファイルは自動的に閉じられます。`r`は読込、`w`は新規作成または上書き、`a`は末尾への追加です。ここではUTF-8を明示します。CSVでは改行をCSVモジュールへ任せるため`newline=""`も指定します。

In [ ]:
with source_path.open("r", encoding="utf-8", newline="") as file:
    text = file.read()

print(text)


## 2.3.3 構造を壊さずにCSVレコードを読む

CSVではフィールド内のコンマを引用符で囲めます。サンプルの2行目の書名にはコンマがありますが、それでも一つの書名です。標準ライブラリの`csv`モジュールなら、この規則を正しく扱えます。

In [ ]:
import csv

with source_path.open("r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    print("Header:", reader.fieldnames)
    raw_rows = list(reader)

for row in raw_rows:
    print(row)


### CSVから読んだ値は、最初はすべて文字列

`DictReader`はヘッダーを辞書のキーにしますが、`false`は自動的に`False`へ変わりません。`bool("false")`は空でない文字列なので`True`です。意味を確認する変換関数を作り、受け入れない値では`ValueError`を送出します。

In [ ]:
def parse_read(value):
    normalised = value.strip().lower()
    if normalised == "true":
        return True
    if normalised == "false":
        return False
    raise ValueError(f"read must be true or false: {value!r}")

print(parse_read(" TRUE "))
print(parse_read("false"))

try:
    parse_read("yes")
except ValueError as error:
    print(type(error).__name__, error)


## 2.3.4 ヘッダー・各行・変換した値を検証する

必要列は`id`、`title`、`read`です。列不足、空のIDや書名、重複ID、不正な真偽値を黙って補正すると、後の処理が誤ったデータで進みます。入力境界で原因を示して止めます。余分な列は、この課題では無視できます。

In [ ]:
REQUIRED_FIELDS = {"id", "title", "read"}

def validate_header(fieldnames):
    actual = set(fieldnames or [])
    missing = REQUIRED_FIELDS - actual
    if missing:
        raise ValueError(f"Missing CSV columns: {sorted(missing)}")

def load_books(path):
    books = []
    seen_ids = set()
    with path.open("r", encoding="utf-8", newline="") as file:
        reader = csv.DictReader(file)
        validate_header(reader.fieldnames)
        for line_number, row in enumerate(reader, start=2):
            book_id = row["id"].strip()
            title = row["title"].strip()
            if not book_id or not title:
                raise ValueError(f"Blank required value on line {line_number}")
            if book_id in seen_ids:
                raise ValueError(f"Duplicate id on line {line_number}: {book_id}")
            books.append({"id": book_id, "title": title, "read": parse_read(row["read"])})
            seen_ids.add(book_id)
    return books

books = load_books(source_path)
print(books)


### 不足するヘッダーも単独で確認する

入力ファイルを壊して試す必要はありません。検証関数へテスト用の見出しを渡し、期待した例外になることを確認できます。

In [ ]:
try:
    validate_header(["id", "title"])
except ValueError as error:
    print(type(error).__name__, error)


## 2.3.5 別ファイルへ保存し、再読込で確認する

教材として渡された`data`のCSVは入力証拠です。変更後の記録は`output`へ保存します。`DictWriter`には出力する列順を明示し、Pythonの真偽値は小文字の`true`または`false`へ戻します。

In [ ]:
def save_books(books, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=["id", "title", "read"])
        writer.writeheader()
        for book in books:
            writer.writerow({
                "id": book["id"],
                "title": book["title"],
                "read": "true" if book["read"] else "false",
            })

source_before = source_path.read_bytes()
updated_books = []
for book in books:
    updated_books.append(book.copy())
updated_books[2]["read"] = True

output_path = source_path.parents[1] / "output" / "lesson23-books-updated.csv"
save_books(updated_books, output_path)
print("Saved:", output_path)


### 保存成功ではなく、再読込して内容を照合する

ファイルが存在するだけでは、列名や型変換が正しいとは限りません。同じ`load_books()`で出力を読み直し、期待したレコードと一致することを確認します。同時に、元CSVのバイト列が変わっていないことも確認します。

In [ ]:
reloaded_books = load_books(output_path)
assert reloaded_books == updated_books
assert source_path.read_bytes() == source_before
assert reloaded_books[2]["read"] is True
print("ROUND TRIP OK")
print("SOURCE PRESERVED")


### `FileNotFoundError`では、探した場所を表示する

ファイル名だけを見ても、Pythonがどのフォルダを基準にしたか分かりません。候補を`resolve()`して表示し、ファイルが配布されているか、名前と大文字小文字が一致するかを順に確認します。

In [ ]:
missing_path = source_path.parent / "missing.csv"
print("Would read:", missing_path.resolve())
print("Exists:", missing_path.exists())


## 2.3.6 統合練習：CSVの往復処理を完成させる

`data/library-books-practice.csv`を読み、`L001`の書名だけを`Python Foundations`へ変更したコピーを`output/lesson23-practice.csv`へ保存してください。元CSVは変更しません。保存後に`load_books()`で読み直し、レコード数、ID順、書名、真偽値が期待どおりであることを`assert`で確認します。

In [ ]:
# ここに応用練習の解答を書きます。


## まとめ

- パス解決、ファイルを開く処理、CSV解析、型変換、検証を分けました。
- カンマを含む引用符付きフィールドを壊さずに読みました。
- 原本を保護し、保存後の再読込まで含む往復処理を検証しました。

## 次のレッスンへ

2.4では、レコード構造、テスト済み関数、入力検証、CSV入出力を組み合わせ、図書台帳を更新するプログラムを作ります。

**学習時間の目安:** 約3時間
